## 1. Imports

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T

# reproducibility
torch.manual_seed(42)
np.random.seed(42)

print('Improted')

Improted


## 2. Load CSV

In [4]:
CSV_PATH = '/kaggle/input/datasets/manjushwarkhairkar/cleaned-fashion-tech-myntra-dataset-with-features/fashion_preprocessed.csv'

df = pd.read_csv(CSV_PATH)

# drop rows where either path is missing
df = df.dropna(subset=['proc_img_path', 'proc_sketch_path']).reset_index(drop=True)

print(f"Total usable samples: {len(df)}")
df[['name', 'proc_img_path', 'proc_sketch_path']].head(3)

Total usable samples: 14311


,name,proc_img_path,proc_sketch_path
0,Khushal K Women Black Ethnic Motifs Printed Ku...,/kaggle/input/datasets/manjushwarkhairkar/outf...,/kaggle/input/datasets/manjushwarkhairkar/outf...
1,InWeave Women Orange Solid Kurta with Palazzos...,/kaggle/input/datasets/manjushwarkhairkar/outf...,/kaggle/input/datasets/manjushwarkhairkar/outf...
2,Anubhutee Women Navy Blue Ethnic Motifs Embroi...,/kaggle/input/datasets/manjushwarkhairkar/outf...,/kaggle/input/datasets/manjushwarkhairkar/outf...


In [5]:
df.sample(3)

,p_id,name,price,colour,brand,img,ratingCount,avg_rating,description,p_attributes,tok_len,attr_keys,img_path,proc_img_path,proc_sketch_path
13322,5829334.0,Roadster Women Mustard Yellow Solid Hooded Swe...,1199.0,Mustard,Roadster,http://assets.myntassets.com/assets/images/582...,5462.0,4.313255,"Mustard yellow solid sweatshirt, has a hood, t...","{'Body Shape ID': '443,424,324', 'Body or Garm...",24,"['Body Shape ID', 'Body or Garment Size', 'Clo...",/kaggle/input/datasets/hiteshsuthar101/myntra-...,/kaggle/input/datasets/manjushwarkhairkar/outf...,/kaggle/input/datasets/manjushwarkhairkar/outf...
8824,18651648.0,Swtantra Blue & Gold-Toned Satin Saree,5499.0,Blue,Swtantra,http://assets.myntassets.com/assets/images/186...,0.0,0.000000,<b> Design Details </b> <ul> <li> Blue and gol...,"{'Blouse': 'NA', 'Blouse Fabric': 'NA', 'Borde...",41,"['Blouse', 'Blouse Fabric', 'Border', 'Care fo...",/kaggle/input/datasets/hiteshsuthar101/myntra-...,/kaggle/input/datasets/manjushwarkhairkar/outf...,/kaggle/input/datasets/manjushwarkhairkar/outf...
10830,13676096.0,Vishudh Off-White & Pink Printed Dupatta,949.0,Off White,Vishudh,http://assets.myntassets.com/assets/images/136...,71.0,3.507042,Off-White and Pink printed Dupatta and has a p...,"{'Border': 'Printed', 'Fabric': 'Pure Cotton',...",18,"['Border', 'Fabric', 'Occasion', 'Ornamentatio...",/kaggle/input/datasets/hiteshsuthar101/myntra-...,/kaggle/input/datasets/manjushwarkhairkar/outf...,/kaggle/input/datasets/manjushwarkhairkar/outf...


## 3. Define transforms

In [13]:
# outfit image — ResNet-18 expects ImageNet normalisation
img_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225])
])

# sketch — single channel, same spatial size
sketch_transform = T.Compose([
    T.Resize((224, 224)),
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),                    # → [0,1]
    T.Normalize(mean=[0.5], std=[0.5])  # → [-1,1]
])

print('Defined')

Defined


## 4. MyntraSketchDataset class

In [14]:
class MyntraSketchDataset(Dataset):

    def __init__(self, dataframe, img_tfm=None, sketch_tfm=None):
        self.df         = dataframe.reset_index(drop=True)
        self.img_tfm    = img_tfm
        self.sketch_tfm = sketch_tfm

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # load outfit image (RGB)
        img    = Image.open(row['proc_img_path']).convert('RGB')
        # load sketch (grayscale)
        sketch = Image.open(row['proc_sketch_path']).convert('RGB')

        if self.img_tfm:
            img    = self.img_tfm(img)
        if self.sketch_tfm:
            sketch = self.sketch_tfm(sketch)

        return {
            'sketch'  : sketch,          # [1, 224, 224]  → CNN encoder input
            'image'   : img,             # [3, 224, 224]  → reconstruction target
            'name'    : row['name'],
            'colour'  : row['colour'],
            'idx'     : idx
        }

print('Class created!')

Class created!


## 5. Test | Val | Split

In [12]:
full_dataset = MyntraSketchDataset(df, img_tfm=img_transform, sketch_tfm=sketch_transform)

total   = len(full_dataset)
n_train = int(0.8 * total)
n_val   = int(0.1 * total)
n_test  = total - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    full_dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

print(f"Train: {len(train_ds)}  |  Val: {len(val_ds)}  |  Test: {len(test_ds)}")

Train: 11448  |  Val: 1431  |  Test: 1432


## 6. Creating DataLoaders

In [19]:
BATCH_SIZE  = 32
NUM_WORKERS = 2
PIN = torch.cuda.is_available()

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")
print(f"Test  batches: {len(test_loader)}")

Train batches: 358
Val   batches: 45
Test  batches: 45


## 7. Sanity checking i.e inspecting one batch

In [20]:
batch = next(iter(train_loader))

print("Sketch tensor shape :", batch['sketch'].shape)
print("Image  tensor shape :", batch['image'].shape)
print("Sketch value range  :", batch['sketch'].min().item(), "→", batch['sketch'].max().item())
print("Image  value range  :", batch['image'].min().item(),  "→", batch['image'].max().item())
print("Sample names:", batch['name'][:3])

Sketch tensor shape : torch.Size([32, 1, 224, 224])
Image  tensor shape : torch.Size([32, 3, 224, 224])
Sketch value range  : -0.8745098114013672 → 1.0
Image  value range  : -2.1179039478302 → 2.640000104904175
Sample names: ['ONLY Women Green Solid Playsuit', 'THE SILHOUETTE STORE Women Pink & Yellow Striped Top with Trousers', 'IX IMPRESSION Orange & Navy Blue Printed Tie-Up Jumpsuit']
